# Notre Dame Lectures 1991 — Q&A Dataset Generation

**Source:** `BuffettNotreDame.pdf` (192 KB, 48 pages)  
**Content:** Three lectures Buffett gave to Notre Dame faculty, MBA students, and undergraduates in Spring 1991. Lightly edited transcript by Whitney Tilson.

Pages 1–9 are a "Highlights" section that duplicates content from the full transcripts — skipped entirely. The three lectures share some repeated material (Company A vs E, See's Candies, Donald Trump), which the assembly notebook's semantic dedup handles.

**Primary labels expected:** Strategy Development (business quality, moats, circle of competence), Psychology (temperament, peer imitation, weakest link), Personal Life (Graham mentorship, early career, Mrs. B stories), Risk Management (leverage dangers, Trump analysis)

In [1]:
import sys, re, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)
import fitz

## Stage 1: Chunking

The PDF has a duplicate "Highlights" section (pages 1–9) followed by three full lecture transcripts starting at "Lecture to Faculty." Chunking skips the highlights, splits on lecture boundaries, strips Tilson's editorial annotations (`[garbled]`, `[Tape ends]`, `[Told Beemer story...]`), and sub-chunks at paragraph boundaries. Validated in `test_notredame_chunking.py`.

In [2]:
def _sub_chunk(text, source_file, section, max_chars=4000, overlap=500):
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="lecture")]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="lecture_subchunk"))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="lecture_subchunk" if sub_chunks else "lecture"))
    return sub_chunks


def chunk_notredame_lectures(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = "".join(pg.get_text() for pg in doc)
    doc.close()

    # Skip Highlights section — duplicates the full transcripts
    faculty_start = full_text.find("Lecture to Faculty")
    if faculty_start == -1:
        faculty_start = full_text.find("Full Transcripts")
    if faculty_start > 0:
        full_text = full_text[faculty_start:]

    # Split into three lectures
    lecture_boundaries = [
        ("Faculty Lecture", r'Lecture to Faculty'),
        ("MBA Lecture", r'Lecture to MBA Students'),
        ("Undergraduate Lecture", r'Lecture to Undergraduate Students'),
    ]

    lectures = []
    for i, (name, pattern) in enumerate(lecture_boundaries):
        match = re.search(pattern, full_text)
        if not match:
            continue
        start = match.start()
        if i + 1 < len(lecture_boundaries):
            next_match = re.search(lecture_boundaries[i + 1][1], full_text)
            end = next_match.start() if next_match else len(full_text)
        else:
            end = len(full_text)
        lectures.append({"name": name, "text": full_text[start:end].strip()})

    all_chunks = []
    editor_stripped = 0

    for lecture in lectures:
        text = lecture["text"]
        lines = text.split('\n')
        clean_lines = []
        for line in lines:
            line = line.strip()
            if len(line) < 20:
                continue
            if re.match(r'^\[(?!Question|From)[^\]]{20,}\]$', line):
                editor_stripped += 1
                continue
            if line.startswith('[Told ') or line.startswith('[Buffett is warning'):
                editor_stripped += 1
                continue
            line = re.sub(r'\[garbled[^\]]*\]', '', line)
            line = re.sub(r'\[Tape (?:ends|flipped|runs out|picks up)[^\]]*\]', '', line)
            line = re.sub(r'\[(?:Pause|Applause|Laughter)\]', '', line)
            line = line.strip()
            if len(line) > 20:
                clean_lines.append(line)

        clean_text = '\n'.join(clean_lines)
        all_chunks.extend(_sub_chunk(clean_text, "notredame_lectures.pdf", lecture["name"]))

    merged = []
    for chunk in all_chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)

    print(f"Lectures: {len(lectures)} | Editor notes stripped: {editor_stripped} | Chunks: {len(merged)}")
    return merged

In [3]:
chunks = chunk_notredame_lectures(Path("../sources/BuffettNotreDame.pdf"))

Lectures: 3 | Editor notes stripped: 11 | Chunks: 35


## Stage 2: Classification

35 chunks through DeepSeek. The faculty lecture should skew toward Strategy Development and Personal Life. The MBA lecture should hit Strategy Development and Psychology heavily. The undergraduate lecture will overlap with the MBA content — duplicates get caught in assembly.

In [5]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "notredame_classified")

Classifying 35 chunks (skipping 0 pre-labeled)
  10/35
  20/35
  30/35
  35/35

Label distribution:
  Strategy Development: 26
  Psychology: 6
  Risk Management: 2
  Personal Life: 1
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\notredame_classified.pkl


In [6]:
dist = Counter(c.label for c in classified if c.label)
print("Label distribution:\n")
for label, count in dist.most_common():
    bar = "█" * count
    print(f"  {label:25s} {count:3d} {bar}")
failed = [c for c in classified if c.label is None]
if failed:
    print(f"\n!! {len(failed)} chunks unclassified")

Label distribution:

  Strategy Development       26 ██████████████████████████
  Psychology                  6 ██████
  Risk Management             2 ██
  Personal Life               1 █


## Stage 3: Q&A Generation

35 chunks × 3 pairs = ~105 candidates. This document uses the updated generation prompt with varied question types (factual, reasoning, process, comparative) and variable answer lengths matched to question complexity.

In [7]:
pairs = await generate_all(classified, n_pairs=3)
save_checkpoint(pairs, "notredame_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")

  5/35 chunks | 15 pairs
  10/35 chunks | 30 pairs
  15/35 chunks | 45 pairs
  20/35 chunks | 60 pairs
  25/35 chunks | 75 pairs
  30/35 chunks | 90 pairs
  35/35 chunks | 105 pairs
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\notredame_pairs_raw.pkl

Total raw pairs: 105


In [8]:
seen_labels = set()
for p in pairs:
    if p.label in seen_labels: continue
    seen_labels.add(p.label)
    print(f"── {p.label} ──")
    print(f"  Q: {p.question}")
    print(f"  A: {p.answer[:250]}...")
    print()

── Strategy Development ──
  Q: According to the passage, what are the two primary tasks Warren Buffett and Charlie Munger assign themselves at Berkshire Hathaway?
  A: Their first task is to identify talented managers and provide an environment where they can operate freely. Their second and only other task is the intelligent deployment of the cash those managers send to headquarters....

── Psychology ──
  Q: According to the passage, what specific psychological trait does Warren Buffett look for in the owners of businesses he acquires, and why is it critical?
  A: Buffett looks for owners who love the business itself, not just the money. He states that after selling their business and becoming very rich, an owner who loves money is 'not of any use to us' because Buffett can't give them enough additional money ...

── Risk Management ──
  Q: What is Warren Buffett's analogy for the hidden, long-term risks in the insurance business, and what does it illustrate about risk management?
 

## Stage 4: Quality Scoring & Filtering

Updated scoring prompt evaluates richness relative to question complexity — short precise answers to factual questions score the same as detailed analytical answers to complex questions.

In [9]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "notredame_pairs_filtered")

  Scored 10/105
  Scored 20/105
  Scored 30/105
  Scored 40/105
  Scored 50/105
  Scored 60/105
  Scored 70/105
  Scored 80/105
  Scored 90/105
  Scored 100/105
  Scored 105/105
Quality filter: 81/105 passed (threshold=0.7)
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\notredame_pairs_filtered.pkl


In [10]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)} / {len(pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(pairs))*100:.1f}%\n")
print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
print(f"Mean:   {statistics.mean(scores):.2f}")
print(f"Median: {statistics.median(scores):.2f}")

Pairs after filtering: 81 / 105 raw
Drop rate: 22.9%

Score range: 0.73 — 0.93
Mean:   0.85
Median: 0.86


## Stage 5: Coverage & Export

This is the final document before assembly. Should contribute to Psychology (temperament, peer imitation, weakest link), Personal Life (Graham mentorship, early career stories, Mrs. B), and Risk Management (leverage, Trump). Strategy Development will add more but is already well past target.

In [11]:
print("Notre Dame contribution:\n")
report = coverage_audit(filtered)

Notre Dame contribution:

  Personal Life               2 pairs ( 2.5%)
    Sublabels hit: habits, personal_values
  Strategy Development       60 pairs (74.1%)
    Sublabels hit: value_investing_framework, margin_of_safety, competitive_moat, business_quality, circle_of_competence, capital_allocation
  Timing                      0 pairs ( 0.0%)
  Risk Management             4 pairs ( 4.9%)
    Sublabels hit: leverage_avoidance, permanent_loss, insurance_float
  Adaptability                0 pairs ( 0.0%)
  Psychology                 15 pairs (18.5%)
    Sublabels hit: temperament, emotional_discipline, contrarian_thinking, patience_psychology, independence, rationality

  Total: 81 pairs


In [12]:
export_csv(filtered, Path("../intermediate/notredame_qa.csv"))
export_detailed(filtered, Path("../intermediate/notredame_qa_detailed.csv"))

Exported 81 pairs to ..\intermediate\notredame_qa.csv
  Strategy Development: 60
  Psychology: 15
  Risk Management: 4
  Personal Life: 2
Detailed export: ..\intermediate\notredame_qa_detailed.csv


In [13]:
label_counts = Counter(p.label for p in filtered)
print(f"FINAL: {len(filtered)} pairs across {len(label_counts)} labels\n")
for label, count in label_counts.most_common():
    best = max((p for p in filtered if p.label == label), key=lambda p: p.composite_score)
    print(f"── {label} ({count} pairs, best: {best.composite_score:.2f}) ──")
    print(f"  Q: {best.question}")
    print(f"  A: {best.answer[:300]}")
    print()

FINAL: 81 pairs across 4 labels

── Strategy Development (60 pairs, best: 0.91) ──
  Q: According to the passage, what is the core of Warren Buffett's investment strategy, and how does he apply it regardless of ownership stake?
  A: The core strategy is to buy pieces of businesses you can understand when they are offered for quite a bit less than they're worth. Buffett applies this principle identically whether buying 100% of a business, like Mrs. B's furniture operation, or a small part of one, like Berkshire's 7% stake in Coc

── Psychology (15 pairs, best: 0.93) ──
  Q: Why does Buffett consider two specific chapters from 'The Intelligent Investor' more important than all other investment writings?
  A: Buffett highlights chapters 8 and 20 of Benjamin Graham's book because they provide the essential 'philosophical bedrock' or correct 'frame of mind' for investing. They contain no specific technical knowledge but instead teach the critical temperament and attitude—particularly toward

In [14]:
# classified = load_checkpoint("notredame_classified")
# pairs = load_checkpoint("notredame_pairs_raw")
# filtered = load_checkpoint("notredame_pairs_filtered")